# 01 Data Exploration

This notebook begins the EDA process for the NSE Stock Risk Intelligence Dashboard.

For now, we will only do three things:

1. Define the objective of EDA
2. Load the database
3. Understand the available tables


## 1. Objective Of EDA

The objective of this EDA is to understand whether the collected NSE stock data is ready for risk analysis.

At this stage, we are not building models and we are not calculating final risk scores. We are only trying to understand the data foundation.

The main questions are:

1. What tables are available in the database?
2. What type of information does each table contain?
3. Which tables will be useful for daily price analysis, sector analysis, intraday analysis, corporate action checks, and refresh tracking?

This step is important because risk analytics should not begin until we know exactly what data we have.

## 2. Load Database

The project database is stored as a SQLite file. We will connect to it directly from the notebook.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


In [ ]:
# If this notebook is opened from the notebooks folder, project root is one folder above.
# If it is opened from the project root, project root is the current folder.
current_path = Path.cwd()
project_root = current_path.parent if current_path.name == 'notebooks' else current_path

database_path = project_root / 'data' / 'database' / 'market_data.db'

database_path

In [ ]:
if not database_path.exists():
    raise FileNotFoundError(f'Database not found: {database_path}')

connection = sqlite3.connect(database_path)

print('Connected to database successfully')

## 3. Understand Available Tables

Now we will list all tables and views inside the database.

A table stores actual data. A view is a saved query that presents data from one or more tables.

In [ ]:
tables = pd.read_sql_query(
    """
    SELECT
        name,
        type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY type, name
    """,
    connection,
)

tables

Next, we will count how many rows each table contains. This gives us a first sense of database size and which tables already have data.

In [ ]:
table_names = tables.loc[tables['type'] == 'table', 'name'].tolist()

row_counts = []

for table_name in table_names:
    count_query = f'SELECT COUNT(*) AS row_count FROM {table_name}'
    row_count = pd.read_sql_query(count_query, connection)['row_count'].iloc[0]
    row_counts.append({'table_name': table_name, 'row_count': row_count})

row_counts = pd.DataFrame(row_counts).sort_values('row_count', ascending=False)

row_counts

Now we will inspect the columns of each table. This tells us what information is available before we start deeper EDA.

In [ ]:
table_columns = []

for table_name in table_names:
    columns = pd.read_sql_query(f'PRAGMA table_info({table_name})', connection)
    columns['table_name'] = table_name
    table_columns.append(columns[['table_name', 'cid', 'name', 'type', 'notnull', 'pk']])

table_columns = pd.concat(table_columns, ignore_index=True)

table_columns

### Table Purpose Summary

| Table | Purpose |
| --- | --- |
| `symbols` | Stock universe, ticker, company name, sector, industry, and index metadata |
| `market_prices` | Daily OHLCV price data for stocks and benchmark |
| `market_intraday_prices` | 30-minute OHLCV data for selected symbols |
| `corporate_actions` | Dividends and splits used to understand adjusted vs raw prices |
| `refresh_runs` | ETL run history and refresh status |
| `refresh_symbol_counts` | Per-symbol row counts for each ETL run |
| `quality_issues` | Data quality issues found during refresh |

At this point we have completed the first EDA step: understanding the database structure.

The next notebook section should start with daily price data overview, but we will only add that after confirming this structure is clear.